In [3]:
import os
from pathlib import Path

import atom3d.datasets as da


In [4]:
par_path =  Path.cwd().parent
print(f"Parent path: {par_path}")

Parent path: /teamspace/studios/this_studio/ai4science/projects/affinitydiff_rl


In [5]:
data_path = os.path.join(par_path, 'data/split-by-sequence-identity-30/data/train')  # noqa: E501
dataset = da.load_dataset(data_path, 'lmdb')

print(f"✓ Successfully loaded {len(dataset)} training examples")
# Expected: ✓ Successfully loaded ~3500 training examples

✓ Successfully loaded 3507 training examples


In [6]:
item = dataset[0]

# --- Target ---
print("=== Target ===")
print(f"PDB ID: item['id'] = {item['id']}")
print(f"Affinity (-log Kd): item['scores']['neglog_aff'] = {item['scores']['neglog_aff']:.2f}  (higher = stronger binding)")

=== Target ===
PDB ID: item['id'] = 1ugx
Affinity (-log Kd): item['scores']['neglog_aff'] = 5.91  (higher = stronger binding)


In [ ]:
import numpy as np

prot = item['atoms_protein']

print("=== Protein (atoms_protein) ===")

protein_coords = prot[['x', 'y', 'z']].values
print(f"\n[x, y, z] → 3D coordinates: shape {protein_coords.shape}")
print(protein_coords[:3])

print(f"\n[element]: {prot['element'].tolist()[:25]}")
print(f"    → atom type (one-hot): {prot['element'].value_counts().to_dict()}")

protein_residues = prot[prot['hetero'] != 'W']['resname'].unique()
print(f"\n[resname] → residue type (one-hot): {len(protein_residues)} unique — {protein_residues[:8]}")

print(f"\n[name] → atom role sample: {prot['name'].value_counts().head(5).to_dict()})")

print(f"\n[bfactor] → flexibility scalar: min={prot['bfactor'].min():.2f}, max={prot['bfactor'].max():.2f}, mean={prot['bfactor'].mean():.2f}")

n_water = (prot['hetero'] == 'W').sum()
n_protein = (prot['hetero'] != 'W').sum()
print(f"\n[hetero == 'W'] → water mask: {n_protein} protein atoms, {n_water} water atoms")

=== Protein (atoms_protein) ===

[x, y, z] → 3D coordinates: shape (9512, 3)
[[36.133999 33.011002 22.528   ]
 [35.917999 33.113998 23.540001]
 [37.096001 32.632    22.414   ]]

[element]: ['N', 'H', 'H', 'H', 'C', 'H', 'H', 'C', 'O', 'N', 'H', 'C', 'H', 'C', 'O', 'C', 'H', 'H', 'C', 'H', 'H', 'C', 'H', 'H', 'C']
    → atom type (one-hot): {'H': 4460, 'C': 2984, 'O': 1352, 'N': 708, 'S': 8}

[resname] → residue type (one-hot): 19 unique — ['GLY' 'LYS' 'ALA' 'PHE' 'ASP' 'THR' 'ILE' 'ARG']

[name] → atom role sample: {'O': 1072, 'C': 592, 'N': 592, 'CA': 592, 'H': 556})

[bfactor] → flexibility scalar: min=0.00, max=60.86, mean=16.68

[hetero == 'W'] → water mask: 9032 protein atoms, 480 water atoms


In [13]:
lig = item['atoms_ligand']

print("=== Ligand (atoms_ligand) ===")

ligand_coords = lig[['x', 'y', 'z']].values
print(f"\n[x, y, z] → 3D coordinates: shape {ligand_coords.shape}")
print(ligand_coords[:3])

print(f"\n[element]: {lig['element'].tolist()[:25]}")
print(f"    → atom type (one-hot): {lig['element'].value_counts().to_dict()}")

print(f"\n[name] → atom names: {lig['name'].tolist()}")

print("\n[bond type / hybridization / aromaticity] → NOT in record")
print("  → obtain via RDKit from SMILES (see GNN Featurization in ATOM3D_dataset.md)")

=== Ligand (atoms_ligand) ===

[x, y, z] → 3D coordinates: shape (27, 3)
[[38.259 34.451 25.431]
 [39.309 34.61  24.349]
 [40.052 35.931 24.535]]

[element]: ['C', 'C', 'C', 'C', 'C', 'C', 'O', 'O', 'O', 'O', 'O', 'O', 'C', 'C', 'O', 'C', 'O', 'C', 'C', 'N', 'C', 'O', 'C', 'C', 'O']
    → atom type (one-hot): {'C': 15, 'O': 11, 'N': 1}

[name] → atom names: ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'O1', 'O2', 'O3', 'O4', 'O5', 'O6', 'C7', 'C8', 'O7', 'C9', 'O8', 'C10', 'C11', 'N1', 'C12', 'O9', 'C13', 'C14', 'O10', 'C15', 'O11']

[bond type / hybridization / aromaticity] → NOT in record
  → obtain via RDKit from SMILES (see GNN Featurization in ATOM3D_dataset.md)


In [15]:
from scipy.spatial.distance import cdist

print("=== Interaction (derived from both) ===")

dist_matrix = cdist(protein_coords, ligand_coords)
print(f"\n[cdist(protein_coords, ligand_coords)] → pairwise distances: shape {dist_matrix.shape}")
print(f"  min={dist_matrix.min():.2f} Å, max={dist_matrix.max():.2f} Å")

cutoff_ang = 5.0
contact_map = dist_matrix < cutoff_ang
print(f"\n[dist < {cutoff_ang} Å] → contact map: {contact_map.sum()} protein-ligand atom pairs in contact")

ligand_centroid = ligand_coords.mean(axis=0)
print(f"\n[ligand centroid] → mean of ligand coordinates: {ligand_centroid}")
pocket_cutoff = 10.0
pocket_dists = np.linalg.norm(protein_coords - ligand_centroid, axis=1)
pocket_mask = pocket_dists < pocket_cutoff
print(f"\n[dist to ligand centroid < {pocket_cutoff} Å] → pocket atoms: {pocket_mask.sum()} / {len(protein_coords)} protein atoms")

=== Interaction (derived from both) ===

[cdist(protein_coords, ligand_coords)] → pairwise distances: shape (9512, 27)
  min=1.87 Å, max=82.35 Å

[dist < 5.0 Å] → contact map: 499 protein-ligand atom pairs in contact

[ligand centroid] → mean of ligand coordinates: [37.00537037 33.40937037 25.72385185]

[dist to ligand centroid < 10.0 Å] → pocket atoms: 200 / 9512 protein atoms
